# 05 — Deploy the SETU-Rail Gradio App on Databricks

**What you will do here**
Deploy the Gradio app (`app.py`) as a Databricks App. The app wires together Dhara (delay model), Vani (Param-1 RAG), Cascade, Drishti, and Analytics.

**Pre-requisites**
- All previous notebooks (01–04 in the repo) ran successfully
- These tables exist: `seturail.rail.gold_features_delay_ml`, `gold_rule_embeddings`, `gold_station_graph`, `vw_delays_by_station`
- These Volume files exist: `/Volumes/seturail/rail/indexes/rules_faiss.index`, `rules_metadata.pkl`, `vani_config.pkl`, `shap_surrogate.pkl`
- MLflow model `setu_dhara_delay_predictor` is registered
- Param-1 GGUF is downloaded to `/Volumes/seturail/rail/models/`

**How to deploy (UI path)**

1. In the left sidebar click **Compute → Apps → Create app**.
2. Choose **Custom**, name it `setu-rail-app`, click **Create app**.
3. Wait until the app is created.
4. Click **Deploy** (top-right) → **Deploy using different source code**.
5. Under Source code, point to this `05_application` folder.
6. Click **Deploy**. Databricks installs from `requirements.txt` and starts the app.
7. When status is **Running**, click **Open** — you'll see the 5 tabs.

**How to deploy (CLI path, optional)**
```bash
databricks apps create setu-rail-app
databricks apps deploy setu-rail-app \
    --source-code-path /Workspace/Users/<you>/setu-rail/05_application
```

**If Param-1 is too heavy for the App container** (the App runs with limited memory): deploy a lighter UI that only handles Dhara + Analytics + Drishti (calls Vani via a notebook endpoint). Most likely it works directly because Q4_K_M Param-1 uses ~2.5 GB.

**Troubleshooting**

| Problem | Fix |
|---|---|
| App stuck in Starting | Open **Logs** from the app page. Most common issue: Param-1 download timeout. Run notebook 03/02 again to pre-cache. |
| `ModuleNotFoundError` on launch | Add the missing package to `requirements.txt` and redeploy. |
| FAISS index not found | Confirm you ran `03_rag_pipeline/01_build_vector_index` and the file is in the Volume. |
| MLflow model load fails | Check `setu_dhara_delay_predictor` exists in Models. Re-run `02_ml_training/01_train_delay_gbt` if needed. |
| IndicTrans2 OOM | Set `INDICTRANS_READY = False` in app — falls back to English-only. Still works for demo. |

In [ ]:
# Quick pre-flight check: run this before clicking Deploy
import os

required_files = [
    "/Volumes/seturail/rail/indexes/rules_faiss.index",
    "/Volumes/seturail/rail/indexes/rules_metadata.pkl",
    "/Volumes/seturail/rail/indexes/vani_config.pkl",
    "/Volumes/seturail/rail/indexes/shap_surrogate.pkl",
]
for f in required_files:
    status = "✓ OK" if os.path.exists(f) else "✗ MISSING"
    print(f"{status}  {f}")

required_tables = [
    "seturail.rail.gold_features_delay_ml",
    "seturail.rail.gold_rule_embeddings",
    "seturail.rail.gold_station_graph",
    "seturail.rail.vw_delays_by_station",
    "seturail.rail.vw_delays_by_month",
    "seturail.rail.vw_weather_delay_correlation",
]
for t in required_tables:
    try:
        n = spark.sql(f"SELECT COUNT(*) n FROM {t}").collect()[0]["n"]
        print(f"✓ OK  {t}  ({n:,} rows)")
    except Exception as e:
        print(f"✗ MISSING  {t}  ({e})")

# Check MLflow registered model
import mlflow
client = mlflow.MlflowClient()
try:
    versions = client.search_model_versions("name='setu_dhara_delay_predictor'")
    if versions:
        print(f"✓ OK  MLflow: setu_dhara_delay_predictor  ({len(versions)} versions)")
    else:
        print("✗ MISSING  MLflow: setu_dhara_delay_predictor  (0 versions)")
except Exception as e:
    print(f"?  MLflow lookup failed: {e}")

Once everything above is ✓, deploy the app via the UI path (Steps 1-7 at the top).

**After deployment** — for the 5-min pitch, have these tabs open in order:
1. Dhara → enter Train 12615, MAS, today's date → show prediction + SHAP
2. Vani → switch to Tamil → paste the test query → show cited answer
3. Cascade → enter Train 12615 at MAS with 60 min delay → show impacted trains
4. Drishti → show action card (the big wow moment)
5. Analytics → click Monthly delay trend + Weather correlation